<a href="https://colab.research.google.com/github/raaghavkk/UG04-NLP-COMM061/blob/main/notebooks/akshyat_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

2.3 Monolingual vs Multilingual
COM3029 NLP Group Coursework


In [ ]:
!pip install transformers datasets peft scikit-learn accelerate

In [ ]:
import torch
import random
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from sklearn.metrics import classification_report, f1_score

seeds = [42, 1234]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

mono_model_name = "meta-llama/Llama-3.2-1B"
multi_model_name = "xlm-roberta-base"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

print(f"Device: {device}")
print(f"Monolingual model: {mono_model_name}")
print(f"Multilingual model: {multi_model_name}")

In [ ]:
# llama 3.2 needs a user agreement (https://huggingface.co/meta-llama/Llama-3.2-1B)
# and a token from settings (https://huggingface.co/settings/tokens) to run.
# the token is then saved in colab secrets as "HF_TOKEN"
# (outputs should already saved so you shouldn't need to)
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get("HF_TOKEN"))

In [ ]:
# Load dataset
ds = load_dataset("surrey-nlp/BESSTIE-CW-26")

train_df = ds["train"].to_pandas()
val_df   = ds["validation"].to_pandas()
test_df  = ds["test"].to_pandas()


for df in [train_df, val_df, test_df]:
    df["Sarcasm"] = df["Sarcasm"].astype(int)

# variety splits for LoRA adapters
train_au = train_df[train_df["variety"] == "en-AU"].reset_index(drop=True)
train_uk = train_df[train_df["variety"] == "en-UK"].reset_index(drop=True)
train_in = train_df[train_df["variety"] == "en-IN"].reset_index(drop=True)
val_au   = val_df[val_df["variety"] == "en-AU"].reset_index(drop=True)
val_uk   = val_df[val_df["variety"] == "en-UK"].reset_index(drop=True)
val_in   = val_df[val_df["variety"] == "en-IN"].reset_index(drop=True)

# test sets
test_au  = test_df[test_df["variety"] == "en-AU"].reset_index(drop=True)
test_uk  = test_df[test_df["variety"] == "en-UK"].reset_index(drop=True)
test_in  = test_df[test_df["variety"] == "en-IN"].reset_index(drop=True)

train_pooled = train_df.reset_index(drop=True)
val_pooled   = val_df.reset_index(drop=True)

print("Train set sizes:")
print("en-AU:", len(train_au))
print("en-UK:", len(train_uk))
print("en-IN:", len(train_in))
print("Pooled:", len(train_pooled))

print("\nSarcasm breakdown:")

au_counts = train_au["Sarcasm"].value_counts()
print("en-AU - Not sarcastic:", au_counts.get(0, 0), "Sarcastic:", au_counts.get(1, 0))

uk_counts = train_uk["Sarcasm"].value_counts()
print("en-UK - Not sarcastic:", uk_counts.get(0, 0), "Sarcastic:", uk_counts.get(1, 0))

in_counts = train_in["Sarcasm"].value_counts()
print("en-IN - Not sarcastic:", in_counts.get(0, 0), "Sarcastic:", in_counts.get(1, 0))

In [ ]:
# dataframe
def tokenise_data(df, tokenizer):
    clean_df = df[["text", "Sarcasm"]].rename(columns={"Sarcasm": "labels"})
    dataset = Dataset.from_pandas(clean_df)

    def process_batch(batch):
        return tokenizer(batch["text"], truncation=True, max_length=128)


    dataset = dataset.map(process_batch, batched=True)

    dataset = dataset.remove_columns(["text"])
    dataset.set_format("torch")

    return dataset

#Macro-F1 score
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    score = f1_score(labels, predictions, average="macro")

    return {"macro_f1": score}

print("Helper functions ready to go")

In [ ]:
lora = LoraConfig(task_type="SEQ_CLS", r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], lora_dropout=0.1, bias="none")

print("lora config ready - r=8, alpha=16")

Monolingual: LLaMA-3.2-1B + LoRA



In [ ]:
def train_lora_adapter(train_df, val_df, variety_name, seed=42):
    set_seed(seed)

    print("")
    print("Training adapter for:", variety_name, "with seed:", seed)

    tokenizer = AutoTokenizer.from_pretrained(MONO_MODEL)
    tokenizer.pad_token = tokenizer.eos_token

    train_dataset = tokenise_data(train_df, tokenizer)
    val_dataset = tokenise_data(val_df, tokenizer)


    base_model = AutoModelForSequenceClassification.from_pretrained(
        MONO_MODEL, num_labels=2, dtype=torch.float16
    )
    base_model.config.pad_token_id = tokenizer.eos_token_id

    lora_config = LoraConfig(task_type="SEQ_CLS", r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], lora_dropout=0.1, bias="none")


    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()
    model.to(DEVICE)


    save_dir = "adapter_" + variety_name + "_seed" + str(seed)

    # standard training settings
    training_args = TrainingArguments(
        output_dir=save_dir,
        num_train_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=0.0002,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=False,
        seed=seed,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics
    )

    trainer.train()

    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)

    print("Finished and saved to:", save_dir)
    return save_dir

print("training function ready")

In [ ]:
# en-AU adapter two runs to confirm stability
adapter_au_run1 = train_lora_adapter(train_au, val_au, "en-AU", seed=SEEDS[0])
adapter_au_run2 = train_lora_adapter(train_au, val_au, "en-AU", seed=SEEDS[1])

In [ ]:
# en-UK adapter
adapter_uk_run1 = train_lora_adapter(train_uk, val_uk, "en-UK", seed=SEEDS[0])
adapter_uk_run2 = train_lora_adapter(train_uk, val_uk, "en-UK", seed=SEEDS[1])

In [ ]:
# en-IN adapter most linguistically distinct, contains code-mixed Hindi-English
# LLaMA has not seen Hindi during pre-training so this is the critical test case
adapter_in_run1 = train_lora_adapter(train_in, val_in, "en-IN", seed=SEEDS[0])
adapter_in_run2 = train_lora_adapter(train_in, val_in, "en-IN", seed=SEEDS[1])

In [ ]:
# function to test the saved lora model
def evaluate_lora(adapter_path, test_df):

    tokenizer = AutoTokenizer.from_pretrained(adapter_path)
    tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForSequenceClassification.from_pretrained(
        MONO_MODEL, num_labels=2, dtype=torch.float16
    )
    base_model.config.pad_token_id = tokenizer.eos_token_id

    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.to(DEVICE)
    model.eval()

    trainer = Trainer(
        model=model,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics
    )

    test_data = tokenise_data(test_df, tokenizer)
    results = trainer.predict(test_data)
    predictions = np.argmax(results.predictions, axis=-1)
    actual_labels = results.label_ids

    score = f1_score(actual_labels, predictions, average="macro")
    report = classification_report(actual_labels, predictions, target_names=["Not Sarcastic", "Sarcastic"], output_dict=True)

    return score, report

print("testing function ready")

In [ ]:
# testing the models from run 1
print("Testing run 1 models...")


model_names = ["en-AU", "en-UK", "en-IN"]
model_paths = [adapter_au_run1, adapter_uk_run1, adapter_in_run1]

data_names = ["en-AU", "en-UK", "en-IN"]
datasets = [test_au, test_uk, test_in]

lora_results_run1 = {}
lora_reports_run1 = {}


for i in range(len(model_names)):
    model = model_names[i]
    path = model_paths[i]


    lora_results_run1[model] = {}
    lora_reports_run1[model] = {}

    # test this model on every dataset
    for j in range(len(data_names)):
        data_name = data_names[j]
        data = datasets[j]

        score, report = evaluate_lora(path, data)

        lora_results_run1[model][data_name] = round(score, 4)
        lora_reports_run1[model][data_name] = report

        print("Model:", model, "-> Data:", data_name, "Score:", round(score, 4))

In [ ]:
# testing the models from run 2
print("\nTesting run 2 models...")

model_paths_run2 = [adapter_au_run2, adapter_uk_run2, adapter_in_run2]

lora_results_run2 = {}

for i in range(len(model_names)):
    model = model_names[i]
    path = model_paths_run2[i]

    lora_results_run2[model] = {}

    for j in range(len(data_names)):
        data_name = data_names[j]
        data = datasets[j]

        score, report = evaluate_lora(path, data)

        lora_results_run2[model][data_name] = round(score, 4)

        print("Run 2 - Model:", model, "-> Data:", data_name, "Score:", round(score, 4))

In [ ]:
# summary tables for both runs
print("run 1 matrix:")
print(pd.DataFrame(lora_results_run1).T)

print()
print("run 2 matrix:")
print(pd.DataFrame(lora_results_run2).T)

print()
print("detailed breakdown for run 1:")


for i in range(len(model_names)):
    model = model_names[i]

    for j in range(len(data_names)):
        data_name = data_names[j]


        report = lora_reports_run1[model][data_name]

        print()
        print("model:", model, "tested on:", data_name)

        # extract not sarcastic scores and round them
        not_sarc_p = round(report["Not Sarcastic"]["precision"], 3)
        not_sarc_r = round(report["Not Sarcastic"]["recall"], 3)
        not_sarc_f1 = round(report["Not Sarcastic"]["f1-score"], 3)
        print("not sarcastic - precision:", not_sarc_p, "recall:", not_sarc_r, "f1:", not_sarc_f1)

        # extract sarcastic scores and round them
        sarc_p = round(report["Sarcastic"]["precision"], 3)
        sarc_r = round(report["Sarcastic"]["recall"], 3)
        sarc_f1 = round(report["Sarcastic"]["f1-score"], 3)
        print("sarcastic     - precision:", sarc_p, "recall:", sarc_r, "f1:", sarc_f1)

        # extract macro average scores and round them
        mac_p = round(report["macro avg"]["precision"], 3)
        mac_r = round(report["macro avg"]["recall"], 3)
        mac_f1 = round(report["macro avg"]["f1-score"], 3)
        print("macro average - precision:", mac_p, "recall:", mac_r, "f1:", mac_f1)

 Multilingual: XLM-RoBERTa-base


In [ ]:
# train multilingual model
def train_xlm(train_df, val_df, seed=42):
    set_seed(seed)

    print("")
    print("training xlm-roberta with seed:", seed)

    tokenizer = AutoTokenizer.from_pretrained(MULTI_MODEL)

    train_dataset = tokenise_data(train_df, tokenizer)
    val_dataset = tokenise_data(val_df, tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(MULTI_MODEL, num_labels=2)
    model.to(DEVICE)

    save_dir = "xlm_roberta_seed_" + str(seed)

    # training settings for xlm-roberta
    training_args = TrainingArguments(
        output_dir=save_dir,
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        learning_rate=0.00002,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        seed=seed,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics
    )

    trainer.train()

    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)

    print("finished training and saved to:", save_dir)

    return save_dir, model, tokenizer

print("xlm training function ready")

In [ ]:
# Train two runs
xlm_path_r1, xlm_model_r1, xlm_tok_r1 = train_xlm(train_pooled, val_pooled, seed=SEEDS[0])
xlm_path_r2, xlm_model_r2, xlm_tok_r2 = train_xlm(train_pooled, val_pooled, seed=SEEDS[1])

In [ ]:
# test xlm model
def evaluate_xlm(model, tokenizer, test_df):
    trainer = Trainer(
        model=model,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics
    )

    test_data = tokenise_data(test_df, tokenizer)

    results = trainer.predict(test_data)

    predictions = np.argmax(results.predictions, axis=-1)
    actual_labels = results.label_ids

    score = f1_score(actual_labels, predictions, average="macro")
    report = classification_report(actual_labels, predictions, target_names=["Not Sarcastic", "Sarcastic"], output_dict=True)

    return score, report

print("xlm testing function ready")

In [ ]:
# test xlm models
print("\ntesting xlm-roberta models...")

xlm_results_r1 = {}
xlm_reports_r1 = {}
xlm_results_r2 = {}

for i in range(len(data_names)):
    data_name = data_names[i]
    data = datasets[i]

    # test run 1
    score_r1, report_r1 = evaluate_xlm(xlm_model_r1, xlm_tok_r1, data)

    # test run 2
    score_r2, report_r2 = evaluate_xlm(xlm_model_r2, xlm_tok_r2, data)

    xlm_results_r1[data_name] = round(score_r1, 4)
    xlm_reports_r1[data_name] = report_r1
    xlm_results_r2[data_name] = round(score_r2, 4)

    print("XLM tested on:", data_name, "| Run 1 score:", round(score_r1, 4), "| Run 2 score:", round(score_r2, 4))

Monolingual vs Multilingual

In [ ]:
# compare the models
print("final comparison:")

lora_r1_au = lora_results_run1["en-AU"]["en-AU"]
lora_r1_uk = lora_results_run1["en-UK"]["en-UK"]
lora_r1_in = lora_results_run1["en-IN"]["en-IN"]

lora_r2_au = lora_results_run2["en-AU"]["en-AU"]
lora_r2_uk = lora_results_run2["en-UK"]["en-UK"]
lora_r2_in = lora_results_run2["en-IN"]["en-IN"]


summary_data = {
    "LLaMA+LoRA Run1": [lora_r1_au, lora_r1_uk, lora_r1_in],
    "LLaMA+LoRA Run2": [lora_r2_au, lora_r2_uk, lora_r2_in],
    "XLM-RoBERTa Run1": [xlm_results_r1["en-AU"], xlm_results_r1["en-UK"], xlm_results_r1["en-IN"]],
    "XLM-RoBERTa Run2": [xlm_results_r2["en-AU"], xlm_results_r2["en-UK"], xlm_results_r2["en-IN"]]
}

comparison = pd.DataFrame(summary_data, index=["en-AU", "en-UK", "en-IN"])
print(comparison)

print()
print("looking at the en-IN results specifically:")

lora_in_avg = (lora_r1_in + lora_r2_in) / 2
xlm_in_avg = (xlm_results_r1["en-IN"] + xlm_results_r2["en-IN"]) / 2

print("llama+lora average:", round(lora_in_avg, 4))
print("xlm-roberta average:", round(xlm_in_avg, 4))

difference = xlm_in_avg - lora_in_avg
print("difference:", round(difference, 4))

if xlm_in_avg > lora_in_avg:
    print("conclusion: xlm-roberta did better. the multilingual training helps with indian english.")
else:
    print("conclusion: llama+lora did better. the specific adapter makes up for the monolingual training.")

print()
print("detailed breakdown for xlm-roberta run 1:")

for i in range(len(data_names)):
    data_name = data_names[i]
    report = xlm_reports_r1[data_name]

    print()
    print("model: xlm-roberta tested on:", data_name)

    not_sarc_p = round(report["Not Sarcastic"]["precision"], 3)
    not_sarc_r = round(report["Not Sarcastic"]["recall"], 3)
    not_sarc_f1 = round(report["Not Sarcastic"]["f1-score"], 3)
    print("not sarcastic - precision:", not_sarc_p, "recall:", not_sarc_r, "f1:", not_sarc_f1)

    sarc_p = round(report["Sarcastic"]["precision"], 3)
    sarc_r = round(report["Sarcastic"]["recall"], 3)
    sarc_f1 = round(report["Sarcastic"]["f1-score"], 3)
    print("sarcastic     - precision:", sarc_p, "recall:", sarc_r, "f1:", sarc_f1)

    mac_p = round(report["macro avg"]["precision"], 3)
    mac_r = round(report["macro avg"]["recall"], 3)
    mac_f1 = round(report["macro avg"]["f1-score"], 3)
    print("macro average - precision:", mac_p, "recall:", mac_r, "f1:", mac_f1)

In [ ]:
# Save best models for Q4 error analysis and Q5 deployment
# Do NOT include these folders in the ZIP — brief says do not submit model weights

print("best lora adapters:")


for i in range(len(model_names)):
    m = model_names[i]
    f1_r1 = lora_results_run1[m][m]
    f1_r2 = lora_results_run2[m][m]
    if f1_r1 >= f1_r2:
        print(m, "- best path:", model_paths[i], "(run 1)")
    else:
        print(m, "- best path:", model_paths_run2[i], "(run 2)")


# check the indian english score directly
if xlm_results_r1["en-IN"] >= xlm_results_r2["en-IN"]:
    print(xlm_path_r1)
else:
    print(xlm_path_r2)

print("\nreminder: share these paths for q4/q5 but do not put them in the zip.")